# Applied Finite Element Analysis — Lecture 11
## Linear elasticity without `LinearProblem`, traction boundary conditions, KSP solvers and preconditioning

## 1. Order of business

1. looking inside the `LinearProblem` functionality to understand what happens when the finite-element linear system is assembled and boundary conditions are applied.

Goal:

- understand how the **stiffness matrix** is assembled;
- understand how **boundary conditions are applied** to the assembled system;
- solve the problem **without relying on the high-level `LinearProblem` solver**;
- introduce **traction boundary conditions**;
- understand the role of **KSP solvers** and **preconditioning**.

The lecture therefore moves from the high-level FEniCSx solve toward the lower-level matrix/vector assembly and PETSc operations.

## 2. What happens after the finite-element forms are assembled?

After discretization, the finite-element problem becomes a linear algebra problem

$$
A u = b,
$$

where:

- $A$ is the assembled stiffness/system matrix,
- $u$ is the vector of unknown degrees of freedom (DOFs), and
- $b$ is the assembled right-hand-side vector.

 **how are prescribed Dirichlet values incorporated into this system?**

## 3. Applying Dirichlet boundary conditions to `A u = b`

Suppose some components of the displacement vector are prescribed. The notes use the example

$$
u_i=s, \qquad i\in I_{bc},
$$

where $I_{bc}$ denotes the set of DOFs on which the boundary condition is imposed.

The remaining DOFs are unknown and must be solved for while respecting the prescribed values.

 If, for example, there are 200 DOFs and 50 are prescribed, the solver should solve for the remaining 150 DOFs while ensuring that those 50 values are exactly the prescribed ones.

### 3.1 Eliminating the known contribution

Consider a partitioned system in which $u_1$ contains prescribed values and $u_2$ contains the unknown values:

$$
\begin{bmatrix}
A_{11} & A_{12}\\
A_{21} & A_{22}
\end{bmatrix}
\begin{bmatrix}
u_1\\u_2
\end{bmatrix}
=
\begin{bmatrix}
b_1\\b_2
\end{bmatrix}.
$$

If $u_1$ is known, say $u_1=s$, its contribution can be moved to the right-hand side. The notes write this idea as

$$
b_2' = b_2-A_{21}u_1,
$$

and, componentwise,

$$
b_i' = b_i-A_{ij}u_j,
$$

for prescribed $u_j$.

**Key idea:** once the prescribed value is known, its contribution to the equations for the remaining unknowns must be accounted for in the RHS.

## 4. How FEniCSx handles the Dirichlet contribution

The matrix does not need to be physically reduced by deleting rows and columns. Instead, the assembled matrix and RHS can be modified while retaining the original data structure.

The important operation introduced is **lifting**. The `apply_lifting` operation uses information about the bilinear form and the prescribed boundary conditions to account for the contribution of the known DOFs in the RHS.



In [ ]:
fem.apply_lifting(b, [a_form], bcs=[bc])

After lifting, the boundary-condition values are imposed on the RHS/vector structure using the boundary-condition information. the assembly process knows where the Dirichlet DOFs are, and the remaining operation is to make the algebraic system consistent with the prescribed values.

## 5. From the high-level `LinearProblem` to explicit assembly

The first classroom code shown for the traction problem uses FEniCSx's high-level `LinearProblem`. In the lecture, this serves as a reference solution before reproducing the operations manually.

In [ ]:
problem = LinearProblem(a, L, bcs=[bc],
                        petsc_options_prefix="L11_trac",
                        petsc_options={"ksp_type": "preonly", "pc_type": "lu"})

uh3 = problem.solve()

diff = uh.x.petsc_vec.copy()
diff.axpy(-1.0, uh3.x.petsc_vec)
print("error norm: ", diff.norm())

## 6. Manual matrix and RHS assembly

The next step is to reproduce the work that `LinearProblem` hides from us.

First, convert the UFL forms into FEniCSx forms and assemble the matrix:

In [ ]:
a_form = fem.form(a)
L_form = fem.form(L)

A = fem.petsc.assemble_matrix(a_form, bcs=[bc])
A.assemble()

b = fem.petsc.assemble_vector(L_form)
fem.apply_lifting(b, [a_form], bcs=[bc])
fem.set_bc(b, [bc])

### What each operation is doing

- `fem.form(a)` prepares the bilinear form for assembly.
- `fem.form(L)` prepares the linear form for assembly.
- `assemble_matrix(...)` constructs the discrete stiffness/system matrix $A$.
- `A.assemble()` completes the PETSc matrix assembly.
- `assemble_vector(...)` constructs the RHS vector $b$.
- `apply_lifting(...)` accounts for the contribution of prescribed Dirichlet values.
- `set_bc(...)` imposes the prescribed values in the assembled RHS representation.

Thus the high-level `LinearProblem` call is being replaced by explicit matrix and vector construction.

## 7. Solving the assembled system with PETSc KSP

Once $A$ and $b$ have been assembled and the boundary conditions have been incorporated, the remaining task is to solve

$$
A u=b.
$$

The  PETSc's **KSP** interface. KSP stands for **Krylov Subspace** methods. Rather than calling a high-level FEniCSx `LinearProblem`, we directly create a PETSc KSP solver, give it the matrix, select the solver type, select a preconditioner, and solve.

In [ ]:
ksp = PETSc.KSP().create(msh.comm)
ksp.setOperators(A)
ksp.setType("preonly")
ksp.getPC().setType("lu")

uh2 = fem.Function(V)
ksp.solve(b, uh2.x.petsc_vec)


- `ksp_type = "preonly"`: do not perform Krylov iterations; use the preconditioner as the solve mechanism.
- `pc_type = "lu"`: use an LU factorization as the preconditioner/direct-solve operation.

The notes describe this as the direct-solve side of the KSP/preconditioner framework.

In [ ]:
diff = uh.x.petsc_vec.copy()
diff.axpy(-1.0, uh2.x.petsc_vec)
print("error norm: ", diff.norm())

The classroom output for the manually assembled system was

```text
error norm: 0.0
```

This verifies, for the quantities compared in class, that the explicitly assembled system and the reference solution agree to the reported norm.

## 8. Why do we need a solver and a preconditioner?

 Finite-element matrices are typically sparse: each basis function interacts only with nearby basis functions, so most matrix entries are zero.

For a large sparse system, explicitly computing

$$
A^{-1}
$$

is not desirable. The lecture notes illustrate the idea that multiplying by $A^{-1}$ would give

$$
A^{-1}Au=A^{-1}b,
$$

but emphasize that we **do not want to explicitly compute the inverse**.

Instead, numerical solvers are used to obtain $u$ efficiently.

## 9. KSP and Krylov-subspace methods

The lecture notes state that PETSc's linear solvers are accessed through **KSP — Krylov Subspace** methods.

The general idea of an iterative Krylov method is to construct progressively better approximations to the solution using information generated from the matrix and RHS, rather than explicitly forming $A^{-1}$.

The particular manual solve demonstrated in class used `preonly` together with an LU preconditioner, so it is useful to distinguish this from a genuinely iterative Krylov solve:

- **Direct/LU approach:** factorize the matrix and use the factorization to solve.
- **Iterative KSP approach:** repeatedly improve an approximate solution, often with the help of a preconditioner.

## 10. Preconditioning

A preconditioner transforms the linear system into a form that is easier for the chosen iterative method to solve.

The lecture notes motivate preconditioning by discussing the sensitivity/conditioning of a matrix. For a suitable matrix, the condition number is associated with the ratio

$$
\kappa(A) \sim \frac{\lambda_{\max}}{\lambda_{\min}},
$$

where the eigenvalues shown in the notes are used to describe the conditioning.

**Lecture takeaway:** we want the numerical solve to be less sensitive to small errors and to converge efficiently. A good preconditioner makes the algebraic problem easier for the solver.

### 10.1 KSP + PC relationship

PETSc separates the linear solver from the preconditioner:

$$
\boxed{\text{KSP solver} + \text{Preconditioner (PC)}}
$$

For example,

```python
ksp.setType("preonly")
ksp.getPC().setType("lu")
```

the preconditioner is  an easier problem/operation in place of directly computing the inverse of $A$.

## 11. Traction boundary conditions

The second major part of the lecture introduces a **traction boundary condition**.

In the weak formulation, a prescribed traction enters through a boundary integral. Starting from the body-force contribution and the boundary term, the RHS has the structure

$$
L(v)=\int_{\Omega} f\cdot v\,dx
+\int_{\Gamma_t} t\cdot v\,ds.
$$

The lecture notes summarize the boundary contribution as the traction term involving the normal vector, i.e. a term associated with $(\sigma\cdot n)\cdot v$.

## 12. Marking the traction facets in FEniCSx



In [ ]:
# Traction problem
right_edges = mesh.locate_entities_boundary(msh, 1, lambda x: np.isclose(x[0], 1.0))
left_edges = mesh.locate_entities_boundary(msh, 1, lambda x: np.isclose(x[0], 0.0))
top_edges = mesh.locate_entities_boundary(msh, 1, lambda x: np.isclose(x[1], 1.0))

traction_facets = np.concatenate([right_edges, left_edges, top_edges])
traction_tags = np.ones(len(traction_facets), dtype=np.int32)

order = np.argsort(traction_facets)
traction_facets = traction_facets[order]
traction_tags = traction_tags[order]

traction_mt = mesh.meshtags(msh, 1, traction_facets, traction_tags)
ds_traction = ufl.Measure("ds", domain=msh, subdomain_data=traction_mt)

### What this code does

The mesh is two-dimensional, so its boundary consists of entities of topological dimension 1 (edges).

The three calls locate:

- the right edge: $x=1$;
- the left edge: $x=0$;
- the top edge: $y=1$.

These arrays are concatenated into `traction_facets`. The facets are sorted with `np.argsort`, and all selected facets receive tag `1`. Finally, `mesh.meshtags` creates the facet tags and `ufl.Measure("ds", ...)` creates a boundary measure that can be restricted using `ds_traction(1)`.

## 13. Traction term in the weak form

The classroom code then defines the outward facet normal and the traction quantity:

In [ ]:
n = ufl.FacetNormal(msh)
t = ufl.dot(sigma(ue), n)
L = ufl.dot(f, v)*ufl.dx + ufl.dot(t, v)*ds_traction(1)

The second term,

$$
\int_{\Gamma_t} t\cdot v\,ds,
$$

is the traction contribution to the RHS.



## 14. Dirichlet condition used with the traction problem

 The variables `u_bc` and `bot_edges` are assumed to have been defined earlier in the notebook/class.

In [ ]:
dofs_D = fem.locate_dofs_topological(V, 1, bot_edges)
bc = fem.dirichletbc(u_bc, dofs_D)

This gives the mixed-boundary-condition setup used in the lecture: a **Dirichlet condition** on one part of the boundary and a **traction condition** on selected boundary facets.

## 16. Manual traction assembly and KSP solve



In [ ]:
a_form = fem.form(a)
L_form = fem.form(L)

A = fem.petsc.assemble_matrix(a_form, bcs=[bc])
A.assemble()

b = fem.petsc.assemble_vector(L_form)
fem.apply_lifting(b, [a_form], bcs=[bc])
fem.set_bc(b, [bc])

ksp = PETSc.KSP().create(msh.comm)
ksp.setOperators(A)
ksp.setType("preonly")
ksp.getPC().setType("lu")

uh2 = fem.Function(V)
ksp.solve(b, uh2.x.petsc_vec)

diff = uh.x.petsc_vec.copy()
diff.axpy(-1.0, uh2.x.petsc_vec)
print("error norm: ", diff.norm())

The classroom output for this manually assembled solve was

```text
error norm: 0.0
```

This is the final numerical check shown in the screenshots.